In [7]:
import numpy as np
from scipy.signal import convolve2d

In [8]:
def apply_conv_layer(images, kernel, mode='same'):
    return np.array([convolve2d(img, kernel, mode=mode, boundary='wrap')
                     for img in images])

def binarize(feature_maps, threshold=None):
    flat = feature_maps.reshape(len(feature_maps), -1)
    t = np.median(flat) if threshold is None else threshold
    return np.where(flat >= t, 1, -1).astype(np.int32)

In [9]:
def images_to_flat(imgs):
    arr = np.array(imgs, dtype=np.int32)
    assert arr.ndim == 3 and arr.shape[1:] == (4, 4), "Expected shape (N, 4, 4)"
    assert set(arr.flatten()).issubset({-1, 1}), "Images must be binary (-1/1)"
    return arr.reshape(len(arr), -1)

def empirical_pmf(patterns):
    keys = [tuple(row) for row in patterns]
    n = len(keys)
    counts = {}
    for k in keys:
        counts[k] = counts.get(k, 0) + 1
    return {k: v / n for k, v in counts.items()}

def entropy(pmf):
    return -sum(p * np.log2(p) for p in pmf.values() if p > 0)

def joint_pmf(pats_A, pats_B):
    assert len(pats_A) == len(pats_B)
    n = len(pats_A)
    jpdf = {}
    for a, b in zip(pats_A, pats_B):
        key = (tuple(a), tuple(b))
        jpdf[key] = jpdf.get(key, 0) + 1
    return {k: v / n for k, v in jpdf.items()}

def mutual_information(pmf_A, pmf_B, jpdf):
    mi = 0.0
    for (a, b), pab in jpdf.items():
        pa = pmf_A.get(a, 0)
        pb = pmf_B.get(b, 0)
        if pa > 0 and pb > 0 and pab > 0:
            mi += pab * np.log2(pab / (pa * pb))
    return mi

def information_summary(flat_A, flat_B, baseline_H, label=""):
    pmf_A = empirical_pmf(flat_A)
    pmf_B = empirical_pmf(flat_B)
    H_A   = entropy(pmf_A)
    H_B   = entropy(pmf_B)
    jpdf  = joint_pmf(flat_A, flat_B)
    MI    = mutual_information(pmf_A, pmf_B, jpdf)
    norm  = MI / max(H_A, H_B) if max(H_A, H_B) > 0 else 0.0

    print(f"\n{'─'*45}")
    print(f"  {label}")
    print(f"{'─'*45}")
    print(f"  H(A)           = {H_A:.4f} bits")
    print(f"  H(B)           = {H_B:.4f} bits")
    print(f"  MI(A; B)       = {MI:.4f} bits")
    print(f"  H(A|B)         = {H_A - MI:.4f} bits")
    print(f"  H(B|A)         = {H_B - MI:.4f} bits")
    print(f"  Normalized MI  = {norm:.4f}  (1.0 = deterministic)")
    print(f"  MI / H(A)      = {MI/baseline_H:.4f}  (fraction of original info preserved)")
    return dict(H_A=H_A, H_B=H_B, MI=MI, norm_MI=norm)

In [10]:
rng = np.random.default_rng(42)

N          = 1000000   # number of base images

In [11]:
KERNELS = {
    # "Identity"              : np.array([[0, 0, 0],
    #                                     [0, 1, 0],
    #                                     [0, 0, 0]], dtype=float),

    "Edge detect (Sobel-x)" : np.array([[-1, 0, 1],
                                        [-2, 0, 2],
                                        [-1, 0, 1]], dtype=float),

    # "Edge detect (Sobel-y)" : np.array([[-1,-2,-1],
    #                                     [ 0, 0, 0],
    #                                     [ 1, 2, 1]], dtype=float),

    # "Blur (box 3×3)"        : np.ones((3, 3), dtype=float) / 9,

    "Laplacian"             : np.array([[ 0,-1, 0],
                                        [-1, 4,-1],
                                        [ 0,-1, 0]], dtype=float),

    # "Random"                : rng.standard_normal((3, 3)),
}

In [12]:
base_images = rng.choice([-1, 1], size=(N, 4, 4))
set_A = base_images.astype(float)


for ROTATION_K in range(1, 4):
    print(f"Rotation: {ROTATION_K*90}°  |  N={N} images  |  Conv output binarized at shared median")
    set_B = np.rot90(base_images, k=ROTATION_K, axes=(-2, -1)).astype(float)

    # ── 1. Raw images (baseline) ───────────────────────────────────────────────────

    flat_A_raw = images_to_flat(set_A.astype(int))
    flat_B_raw = images_to_flat(set_B.astype(int))
    baseline   = information_summary(flat_A_raw, flat_B_raw, baseline_H=1.0, label="RAW IMAGES (baseline)")

    # ── 2. After each conv kernel ──────────────────────────────────────────────────

    results = {"Baseline (raw)": baseline}

    for name, kernel in KERNELS.items():
        feat_A = apply_conv_layer(set_A, kernel, mode='same')
        feat_B = apply_conv_layer(set_B, kernel, mode='same')

        shared_threshold = np.median(np.concatenate([feat_A.ravel(), feat_B.ravel()]))
        q_A = binarize(feat_A, threshold=shared_threshold)
        q_B = binarize(feat_B, threshold=shared_threshold)

        res = information_summary(q_A, q_B, baseline_H=baseline["H_A"], label=f"AFTER CONV — {name}")
        results[name] = res

    # ── 3. Summary table ───────────────────────────────────────────────────────────

    print(f"\n{'═'*55}")
    print(f"  SUMMARY — MI preserved vs. original H(A) = {baseline['H_A']:.4f} bits")
    print(f"{'═'*55}")
    print(f"  {'Kernel':<25} {'MI':>8}  {'MI/H(A)':>10}  {'Norm MI':>10}")
    print(f"  {'-'*50}")
    for name, r in results.items():
        mi_ratio = r["MI"] / baseline["H_A"]
        print(f"  {name:<25} {r['MI']:>8.4f}  {mi_ratio:>10.4f}  {r['norm_MI']:>10.4f}")

Rotation: 90°  |  N=1000000 images  |  Conv output binarized at shared median

─────────────────────────────────────────────
  RAW IMAGES (baseline)
─────────────────────────────────────────────
  H(A)           = 15.9523 bits
  H(B)           = 15.9523 bits
  MI(A; B)       = 15.9523 bits
  H(A|B)         = 0.0000 bits
  H(B|A)         = 0.0000 bits
  Normalized MI  = 1.0000  (1.0 = deterministic)
  MI / H(A)      = 15.9523  (fraction of original info preserved)

─────────────────────────────────────────────
  AFTER CONV — Edge detect (Sobel-x)
─────────────────────────────────────────────
  H(A)           = 9.0392 bits
  H(B)           = 9.0410 bits
  MI(A; B)       = 4.6192 bits
  H(A|B)         = 4.4200 bits
  H(B|A)         = 4.4218 bits
  Normalized MI  = 0.5109  (1.0 = deterministic)
  MI / H(A)      = 0.2896  (fraction of original info preserved)

─────────────────────────────────────────────
  AFTER CONV — Laplacian
─────────────────────────────────────────────
  H(A)         

X = # set of images sampled from a constant distribution
A = f(X)
B = f(gx)

H(X) = 16
H(gX) = 16
H(X) = H(gX) 
gX = X

A = f(X), if f is injective H(A) = H(X)

MI(A;B) < H(A) and MI(A;B) < H(B)

if f destroys information, it identifies traits about X. You can't precisely return to the original image.